In [25]:
#openwebtext datasetinin yüklenmesi ve kontrol edilmesi bloğu
from datasets import load_dataset
dataset = load_dataset("Skylion007/openwebtext", split="train")
print(dataset[0]['text'][:500])

C:\Users\Victus\Desktop\YİĞİT EFE\test\fcc-gpt-course\cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated the area, saying it was concerned about security.

The decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.

CNN initially reported, based on conversations with some of the doctors, that the United Nations ordered the B


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 64 #eldeki cümleyi 512 erli işleyecek
batch_size = 8 #paralel olarak kaç batchi inceleyeceğiz
max_iters = 500
learning_rate = 2e-5
eval_iters = 100
n_embd = 384
n_head = 8
n_layer = 8
dropout = 0.2

cuda


In [27]:
import tiktoken
import torch
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab 
encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: enc.decode(l)
text_data = " ".join([dataset[i]['text'] for i in range(10000)])
data = torch.tensor(encode(text_data), dtype=torch.long)

In [29]:
#sohbet dataseti her zaman yükleme
import json
import torch
import tiktoken

enc = tiktoken.get_encoding("gpt2")
dosya_yolu = "alpaca_data.json"
with open(dosya_yolu, 'r', encoding='utf-8') as f:
    alpaca_raw = json.load(f)
def format_alpaca_prompt(sample):
    if sample.get("input", "").strip() != "":
        return f"### Talimat:\n{sample['instruction']}\n\n### Girdi:\n{sample['input']}\n\n### Cevap:\n{sample['output']}<|endoftext|>"
    else:
        return f"### Talimat:\n{sample['instruction']}\n\n### Cevap:\n{sample['output']}<|endoftext|>"
all_tokens = []
for sample in alpaca_raw:
    formatted_text = format_alpaca_prompt(sample)
    tokens = enc.encode(formatted_text, allowed_special={"<|endoftext|>"})
    all_tokens.extend(tokens)

chat_data = torch.tensor(all_tokens, dtype=torch.long)
print(f"Sohbet eğitimine hazır toplam token sayısı: {len(chat_data)}")
n_chat = int(0.9 * len(chat_data))
train_data = chat_data[:n_chat]
val_data = chat_data[n_chat:]

print(f"Eğitim sohbet verisi: {len(train_data)} token")
print(f"Test sohbet verisi: {len(val_data)} token")

Sohbet eğitimine hazır toplam token sayısı: 4856457
Eğitim sohbet verisi: 4370811 token
Test sohbet verisi: 485646 token


In [ ]:
n = int(0.9*len(data))#datayı parça parça koyuyoruz çünkü hepsini tek koyarsak onu ezberler ve sadece onu çıkarmaya başlar ama %80 bi yerde %20 bi yerde ezberlerlerse birbirinden farklı jenerasyonlar elde edeeriz bize dökümanın aynısını değil döküman "gibi" çıktılar üretir
train_data = data[:n]
val_data = data[n:]


In [31]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size]for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    x,y =x.to(device),y.to(device)
    return x,y

In [ ]:
x = train_data[:block_size] #block size kadar bir alanı alırız
y = train_data[1:block_size+1]# x in 1 adım kaydırılmış halidirmodelin tahmin etmeye çalışacağı targetler burda tutulur

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("input",context,"iken targetimiz",target)

In [33]:
@torch.no_grad()
def estimate_loss():
    out ={}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits,loss = model(X ,Y)
            losses[k] = loss.mean()
        out[split] = losses.mean()
    model.train()
    return out

In [37]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
class Block(nn.Module):
    def __init__(self,n_embd,n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        #Karpathy mimarisinde residual bağlantılar genelde LayerNorm'dan sonra toplanır
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
        
class GPTLanguageModel(nn.Module): 
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.lm_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
    def forward(self, index, targets=None):
        B,T = index.shape

        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.lm_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape 
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
        
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

In [ ]:
model = GPTLanguageModel(vocab_size)
model.load_state_dict(torch.load('ai_save.pt', map_location=device))
model = model.to(device)

In [ ]:
model.load_state_dict(torch.load('ai_save_alpaca.pt', map_location=device))
model = model.to(device)

In [59]:
optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
        torch.save(model.state_dict(), 'ai_save_alpaca.pt')
    xb,yb = get_batch('train')
    
    #kaybı hesapla
    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 3.749, val loss: 3.800
step: 100, train loss: 3.749, val loss: 3.812
step: 200, train loss: 3.720, val loss: 3.760
step: 300, train loss: 3.673, val loss: 3.773
step: 400, train loss: 3.669, val loss: 3.757
3.7049951553344727


In [93]:
context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars= decode(model.generate(context,max_new_tokens=500)[0].tolist())
print(generated_chars)

!
                    
                    u' on apt.     start laughing, man walked away.<|endoftext|>### Talimat:
The ardent being demagogiced from around the Belt of the following Interstruction campaign.

### Girdi:
The walls in the summer are the again mid-2017 crowd. After minutes, the story was located in the meaningful manner of the last four Deadpool.

### Cevap:
I'm the occasion next to the below evening I had been waiting. I'm member of a teacher with the friend I loved. I don't let at all myself.

After my friend, I English, I know I am on my house.<|endoftext|>### Talimat:
Given the words He likes to be so excited about the story.

### Cevap:
He's showing her couple's courage to stay up there.

### Cevap:
He always knew how to write a text essay. He'dynamic excited to go to the hustling.
He smartscript: create a new representation. 

Is it great?

4. 11.00! B.
7.1975cript: What would it be?<|endoftext|>### Talimat:
Write a book about the sources such as a character (Make t

In [ ]:
import torch
import tiktoken

model.eval()
enc = tiktoken.get_encoding("gpt2")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

sorum = "Develop a plan to reduce electricity usage in a home." 

formatted_prompt = f"### Talimat:\n{sorum}\n\n### Cevap:\n"

context_tokens = enc.encode(formatted_prompt)
context = torch.tensor([context_tokens], dtype=torch.long, device=device)

with torch.no_grad():
    generated_tokens = model.generate(context, max_new_tokens=50)[0].tolist()

tam_cevap = enc.decode(generated_tokens)

asistan_cevabi = tam_cevap.split("### Cevap:\n")[-1]

print("Asistan: " + asistan_cevabi)

Asistan: The Orange control of Canada is about 10% that
